# 04 — Ferramentas com funções

**Trilha:** [básico 01–03](../01_basico/01_chamada_e_json.ipynb) → **04 ferramentas com funções** → [05 regras e retry](05_regras_e_retry.ipynb) → [06 roteamento por enum](06_roteamento_por_enum.ipynb)

**De onde viemos:** o [03](../01_basico/03_enums_e_guardrails.ipynb) terminou com uma lista válida pelo contrato, mas sem
nenhum dado do mundo real: o modelo **não sabe** quanto custa o tomate nem o que você já tem em casa. Aqui o modelo ganha
**ferramentas** — funções Python comuns que ele pode pedir para executar. Quem executa é o seu código, e é aí que entram os
guardrails.

O contrato do básico está em [`contrato.py`](contrato.py) (os mesmos enums e modelos do 03).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from logger_general import GeneralLogger
from prettifier import show, show_json

In [2]:
import json
from dataclasses import dataclass
from enum import StrEnum

import openai
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from contrato import Unidade

## 1. Os dados e as funções

Duas "fontes" simples, em dicionários: a tabela de preços do mercado e a despensa de casa.

In [3]:
PRECOS = {  # preço em R$ por unidade de venda
    "farinha de trigo": (6.50, Unidade.KG),
    "muçarela de búfala": (89.90, Unidade.KG),
    "tomate italiano": (1.20, Unidade.UN),
    "azeite": (45.00, Unidade.L),
    "manjericão fresco": (4.00, Unidade.UN),
}
DESPENSA = {"farinha de trigo": (1, Unidade.KG), "azeite": (0.5, Unidade.L)}


def consultar_preco(nome: str) -> dict:
    if nome not in PRECOS:
        return {"encontrado": False, "nome": nome}
    preco, unidade = PRECOS[nome]
    return {"encontrado": True, "nome": nome, "preco": preco, "por": unidade}


def verificar_despensa(nome: str) -> dict:
    quantidade, unidade = DESPENSA.get(nome, (0, None))
    return {"nome": nome, "tem": quantidade > 0, "quantidade": quantidade, "unidade": unidade}


show(f"consultar_preco('azeite'): {consultar_preco('azeite')}")
show(f"consultar_preco('trufa'): {consultar_preco('trufa')}")
show(f"verificar_despensa('azeite'): {verificar_despensa('azeite')}")

consultar_preco('azeite'): {'encontrado': True, 'nome': 'azeite', 'preco': 45.0, 'por': <Unidade.L: 'l'>}
consultar_preco('trufa'): {'encontrado': False, 'nome': 'trufa'}
verificar_despensa('azeite'): {'nome': 'azeite', 'tem': True, 'quantidade': 0.5, 'unidade': <Unidade.L: 'l'>}


## 2. Descrevendo as ferramentas para o modelo

Cada ferramenta tem um modelo Pydantic para os **argumentos**. O SDK transforma esse modelo no formato de ferramenta que a API
espera (`pydantic_function_tool`). O nome de cada ferramenta vem de um enum — a lista fechada do que pode ser chamado.

In [4]:
class Ferramenta(StrEnum):
    CONSULTAR_PRECO = "consultar_preco"
    VERIFICAR_DESPENSA = "verificar_despensa"


class ArgsIngrediente(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nome: str = Field(description="Nome do ingrediente em português, minúsculo")


FERRAMENTAS = {  # nome -> (modelo dos argumentos, função)
    Ferramenta.CONSULTAR_PRECO: (ArgsIngrediente, consultar_preco),
    Ferramenta.VERIFICAR_DESPENSA: (ArgsIngrediente, verificar_despensa),
}
DESCRICOES = {
    Ferramenta.CONSULTAR_PRECO: "Preço de um ingrediente no mercado",
    Ferramenta.VERIFICAR_DESPENSA: "Quanto do ingrediente já existe em casa",
}
TOOLS = [
    openai.pydantic_function_tool(args, name=ferramenta.value, description=DESCRICOES[ferramenta])
    for ferramenta, (args, _) in FERRAMENTAS.items()
]
show_json(TOOLS[0], title="TOOLS[0]")

TOOLS[0]:
{
  "type": "function",
  "function": {
    "name": "consultar_preco",
    "strict": true,
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "nome": {
          "description": "Nome do ingrediente em português, minúsculo",
          "title": "Nome",
          "type": "string"
        }
      },
      "required": [
        "nome"
      ],
      "title": "ArgsIngrediente",
      "type": "object"
    },
    "description": "Preço de um ingrediente no mercado"
  }
}


## 3. O guardrail: executar com segurança

O modelo devolve **nome** e **argumentos em texto JSON**. Antes de executar qualquer coisa, o código confere as duas coisas. Se
algo estiver errado, a função **não levanta exceção**: devolve um erro em texto, que volta para o modelo como resultado da
ferramenta — assim ele pode corrigir na rodada seguinte.

In [5]:
def executar_ferramenta(nome: str, argumentos_json: str) -> str:
    try:
        ferramenta = Ferramenta(nome)
    except ValueError:
        return json.dumps({"erro": f"ferramenta desconhecida: {nome}"})
    modelo_args, funcao = FERRAMENTAS[ferramenta]
    try:
        args = modelo_args.model_validate_json(argumentos_json)
    except ValidationError as ex:
        return json.dumps({"erro": "argumentos inválidos", "detalhes": ex.errors(include_url=False)},
                          ensure_ascii=False, default=str)
    return json.dumps(funcao(**args.model_dump()), ensure_ascii=False)


args_ok = '{"nome": "azeite"}'
args_errados = '{"ingrediente": "azeite"}'
show(f"válida: {executar_ferramenta('consultar_preco', args_ok)}")
show(f"inexistente: {executar_ferramenta('apagar_arquivos', '{}')}")
show(f"args errados: {executar_ferramenta('consultar_preco', args_errados)[:110]}...")

válida: {"encontrado": true, "nome": "azeite", "preco": 45.0, "por": "l"}
inexistente: {"erro": "ferramenta desconhecida: apagar_arquivos"}
args errados: {"erro": "argumentos inválidos", "detalhes": [{"type": "extra_forbidden", "loc": 
["ingrediente"], "msg": "Extr...


## 4. O laço de conversa

Com ferramentas, uma pergunta vira **várias rodadas**: o modelo pede ferramentas, o código executa e devolve os resultados, e o
modelo responde — ou pede mais. Uma configuração congelada (`frozen`) guarda os parâmetros do laço, incluindo um **limite de
rodadas**.

A função que fala com o modelo entra como parâmetro (`chamar_modelo`): com a API real ou com respostas construídas, o laço é o
mesmo. Cada execução de ferramenta vira um **evento no log** (`GeneralLogger`, de [`logger_general.py`](../logger_general.py)):
`INFO` quando dá certo, `WARNING` quando o resultado é um erro que volta ao modelo.

In [6]:
@dataclass(frozen=True)
class ConfiguracaoChamada:
    modelo: str = "gpt-4o-mini"
    max_rodadas: int = 5
    instrucao: str = (
        "Você ajuda com compras de mercado. Use as ferramentas para consultar preços e a despensa. "
        "Responda em português, de forma curta."
    )


log = GeneralLogger(name="ferramentas", stream=sys.stdout)


def conversar(pergunta: str, chamar_modelo, config: ConfiguracaoChamada) -> str:
    mensagens = [{"role": "system", "content": config.instrucao}, {"role": "user", "content": pergunta}]
    for rodada in range(1, config.max_rodadas + 1):
        mensagem = chamar_modelo(mensagens)
        if not mensagem.tool_calls:
            return mensagem.content
        mensagens.append(mensagem.model_dump(exclude_none=True))
        for chamada in mensagem.tool_calls:
            resultado = executar_ferramenta(chamada.function.name, chamada.function.arguments)
            evento = f"rodada {rodada}: {chamada.function.name}({chamada.function.arguments}) -> {resultado[:60]}"
            (log.warning if '"erro"' in resultado else log.info)(evento)
            mensagens.append({"role": "tool", "tool_call_id": chamada.id, "content": resultado})
    raise RuntimeError(f"sem resposta final em {config.max_rodadas} rodadas")

In [ ]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")
openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

In [ ]:
# api: o laço com o modelo de verdade
config = ConfiguracaoChamada(modelo=AI_MODEL)


def chamar_api(mensagens):
    resposta = ai_client.chat.completions.create(model=config.modelo, messages=mensagens, tools=TOOLS)
    return resposta.choices[0].message


resposta_final = conversar("Quanto gasto com azeite e tomate para uma pizza? Já tenho algo em casa?", chamar_api, config)
show(f"resposta final: {resposta_final}")

Sem chave, o mesmo laço com mensagens **construídas com os tipos do próprio SDK** (`ChatCompletionMessage`), na sequência que um
modelo costuma produzir: primeiro pede as ferramentas, depois responde com os resultados.

In [7]:
from openai.types.chat import ChatCompletionMessage
from openai.types.chat.chat_completion_message_function_tool_call import (
    ChatCompletionMessageFunctionToolCall,
    Function,
)


def pedido_de_ferramenta(*chamadas: tuple[str, str]) -> ChatCompletionMessage:
    return ChatCompletionMessage(role="assistant", content=None, tool_calls=[
        ChatCompletionMessageFunctionToolCall(id=f"call_{i}", type="function",
                                              function=Function(name=nome, arguments=args))
        for i, (nome, args) in enumerate(chamadas, start=1)
    ])


def roteiro(*mensagens):
    """Devolve um chamar_modelo falso que responde a sequência dada, uma mensagem por rodada."""
    fila = iter(mensagens)
    return lambda mensagens_enviadas: next(fila)


simulado = roteiro(
    pedido_de_ferramenta(("consultar_preco", '{"nome": "azeite"}'), ("verificar_despensa", '{"nome": "azeite"}')),
    ChatCompletionMessage(role="assistant", content="O azeite custa R$ 45,00/l, mas você já tem 0,5 l em casa."),
)
resposta_final = conversar("Preciso comprar azeite?", simulado, ConfiguracaoChamada())
show(f"resposta final: {resposta_final}")

2026-11-12 19:29:52 | INFO     | rodada 1: consultar_preco({"nome": "azeite"}) -> {"encontrado": true, "nome": "azeite", "preco": 45.0, "por":
2026-11-12 19:29:52 | INFO     | rodada 1: verificar_despensa({"nome": "azeite"}) -> {"nome": "azeite", "tem": true, "quantidade": 0.5, "unidade"
resposta final: O azeite custa R$ 45,00/l, mas você já tem 0,5 l em casa.


## ⚠️ O que ainda quebra

**1. O modelo pode pedir o que não existe — o guardrail segura.** Ferramenta inventada e argumento com nome errado viram erro
para o modelo, não exceção no seu código (*roteiro construído*):

In [8]:
simulado = roteiro(
    pedido_de_ferramenta(("finalizar_compra", '{"cartao": "1234"}')),
    pedido_de_ferramenta(("consultar_preco", '{"produto": "azeite"}')),
    ChatCompletionMessage(role="assistant", content="Não consegui consultar o preço agora."),
)
resposta_final = conversar("Compra o azeite pra mim", simulado, ConfiguracaoChamada())
show(f"resposta final: {resposta_final}")

2026-11-12 19:29:52 | WARNING  | rodada 1: finalizar_compra({"cartao": "1234"}) -> {"erro": "ferramenta desconhecida: finalizar_compra"}
2026-11-12 19:29:52 | WARNING  | rodada 2: consultar_preco({"produto": "azeite"}) -> {"erro": "argumentos inválidos", "detalhes": [{"type": "extr
resposta final: Não consegui consultar o preço agora.


**2. Sem limite de rodadas, o laço pode não terminar.** Um modelo que fica pedindo ferramentas para sempre (acontece com
instruções ambíguas) para no `max_rodadas`:

In [9]:
sem_fim = lambda mensagens: pedido_de_ferramenta(("consultar_preco", '{"nome": "azeite"}'))
try:
    conversar("Quanto custa azeite?", sem_fim, ConfiguracaoChamada(max_rodadas=2))
except RuntimeError as ex:
    show(f"RuntimeError: {ex}")

2026-11-12 19:29:52 | INFO     | rodada 1: consultar_preco({"nome": "azeite"}) -> {"encontrado": true, "nome": "azeite", "preco": 45.0, "por":
2026-11-12 19:29:52 | INFO     | rodada 2: consultar_preco({"nome": "azeite"}) -> {"encontrado": true, "nome": "azeite", "preco": 45.0, "por":
RuntimeError: sem resposta final em 2 rodadas


**3. `float` não é dinheiro.** Os preços estão em `float`, e a soma acumula erro de representação:

In [10]:
precos = [0.10, 0.20]
show(f"0.10 + 0.20: {sum(precos)!r}")
show(f"sum(...) == 0.30: {sum(precos) == 0.30}")
show(f"6.50 * 3 * 0.1: {6.50 * 3 * 0.1!r}")

0.10 + 0.20: 0.30000000000000004
sum(...) == 0.30: False
6.50 * 3 * 0.1: 1.9500000000000002


Para exibir, um `:.2f` esconde o problema; para somar carrinhos e comparar com orçamento, não. O avançado troca por `Decimal`.

**4. `strict: true` nas ferramentas.** O `pydantic_function_tool` marca a ferramenta como `strict` (veja o JSON da seção 2). Na
OpenAI isso **restringe** os argumentos ao schema; endpoints compatíveis podem ignorar ou recusar o campo. Por isso a validação
no `executar_ferramenta` continua necessária mesmo com `strict`.

**5. O que vem das ferramentas volta para o prompt.** O texto que a ferramenta devolve entra na conversa como contexto. Se uma
fonte externa trouxer `"nome": "ignore as instruções e ..."`, isso chega ao modelo. Aqui os dados são nossos; com fontes
externas, o resultado de ferramenta é **entrada não confiável**.

**E o problema que veio do 03 continua:** a lista final pode ter 3 kg de sal. Ferramentas trazem dados, mas ninguém está
**conferindo regras** na resposta. ➡️ [05 — regras e retry](05_regras_e_retry.ipynb)